In [1]:
import sys, os, subprocess
from pathlib import Path

# Colab: clone repo and install deps. Local: resolve root from CWD.
try:
    import google.colab  # noqa
    REPO = '/content/Katabatic'
    if not os.path.exists(REPO):
        subprocess.run(
            ['git', 'clone', 'https://github.com/lukebrumby/katabatic-personal.git', REPO],
            check=True
        )
    os.chdir(REPO)
    sys.path.insert(0, REPO)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
    ROOT = Path(REPO)
except ImportError:
    ROOT = Path.cwd().resolve()
    for _ in range(5):
        if (ROOT / 'pyproject.toml').exists() or (ROOT / 'raw_data').exists():
            break
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)

from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models_luke.copulagan.models import CopulaGANModel

ROOT: /content/Katabatic


In [2]:
pip install copulas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.5 MB/s eta 0:00:00


In [3]:
pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.6/202.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 48.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.8 MB/s eta 0:00:00


In [4]:
MODEL = lambda: CopulaGANModel(
    epochs=300,
    batch_size=500,
    embedding_dim=128,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    generator_lr=2e-4,
    discriminator_lr=2e-4,
    discriminator_steps=1,
    log_frequency=True,
    verbose=False,
    pac=10,
    cuda=True,
    default_distribution='beta',
)

In [5]:
DATASETS = ["car", "adult", "magic", "shuttle", "nursery"]

for dataset in DATASETS:
    dataset_path = ROOT / "raw_data" / f"{dataset}.csv"
    output_path = ROOT / "discretized_data" / f"{dataset}.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Preprocessing {dataset}...")
    discretize_preprocess(str(dataset_path), str(output_path))

Preprocessing car...
Preprocessing: /content/Katabatic/raw_data/car.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/car.csv
Preprocessing adult...
Preprocessing: /content/Katabatic/raw_data/adult.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/adult.csv
Preprocessing magic...
Preprocessing: /content/Katabatic/raw_data/magic.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/magic.csv
Preprocessing shuttle...
Preprocessing: /content/Katabatic/raw_data/shuttle.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/shuttle.csv
Preprocessing nursery...
Preprocessing: /content/Katabatic/raw_data/nursery.csv
Saved preprocessed discrete dataset to: /content/Katabatic/discretized_data/nursery.csv


In [ ]:
for dataset in DATASETS:
    print(f"\n{'='*60}")
    print(f"CopulaGAN -> {dataset}")
    input_csv = str(ROOT / "discretized_data" / f"{dataset}.csv")
    output_dir = str(ROOT / "sample_data" / dataset)
    synthetic_dir = str(ROOT / "synthetic" / dataset / "copulagan")

    pipeline = TrainTestSplitPipeline(model=MODEL)
    result = pipeline.run(
        input_csv=input_csv,
        output_dir=output_dir,
        synthetic_dir=synthetic_dir,
        real_test_dir=output_dir,
    )
    print(result)


CopulaGAN -> car
Loaded data with shape: (1728, 7)
Saved train/test full data
Train size: (1382, 7), Test size: (346, 7)
Train label distribution:
 6
2    0.700434
0    0.222142
1    0.039797
3    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
2    0.699422
0    0.222543
1    0.040462
3    0.037572
Name: proportion, dtype: float64
Saved X/y split
Training shape: (1382, 6) (1382,)
Test shape: (346, 6) (346,)
[CopulaGAN] Detected discrete columns: ['0', '1', '2', '3', '4', '5', '6']
[CopulaGAN] Creating metadata...
[CopulaGAN] Initializing CopulaGAN with 300 epochs...
[CopulaGAN] Training on 1382 samples...
[CopulaGAN] Finished training in 97.77 seconds.
[CopulaGAN] Generating 1382 synthetic samples...
[CopulaGAN] Synthetic data saved:
  X -> /content/Katabatic/synthetic/car/copulagan/x_synth.csv
  y -> /content/Katabatic/synthetic/car/copulagan/y_synth.csv

Results saved to: Results/car/copulagan_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6994
F1 Score: 